In [5]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pyproj folium "shapely>=2.0"

DATA = '/content/drive/MyDrive/hmm-map-data'
!unzip -o {DATA}/road_network.zip -d {DATA}
!ls {DATA}

Mounted at /content/drive
unzip:  cannot find or open /content/drive/MyDrive/hmm-map-data/road_network.zip, /content/drive/MyDrive/hmm-map-data/road_network.zip.zip or /content/drive/MyDrive/hmm-map-data/road_network.zip.ZIP.
candidates.pkl	gps.pkl			road_network.txt  truth_per_point.pkl
gps_data.txt	ground_truth_route.txt	roads.pkl


In [6]:
for name in ['gps_data.txt', 'road_network.txt', 'ground_truth_route.txt']:   # fix names if different
    print('=====', name)
    with open(f'{DATA}/{name}') as f:
        for _ in range(3):
            print(f.readline()[:300])

===== gps_data.txt
Date (UTC)	Time (UTC)	Latitude	Longitude		

17-Jan-2009	20:27:37	47.66748333	-122.1070833		

17-Jan-2009	20:27:38	47.6675	-122.1070667		

===== road_network.txt
Edge ID	From Node ID	To Node ID	Two Way	 Speed (m/s)	Vertex Count	LINESTRING()

883991900000	883991900000	883991900001	1	22.22222222	10	LINESTRING(-122.732318937778 47.8899192810059, -122.732139229774 47.8903403878212, -122.731820046902 47.8910404443741, -122.731310427189 47.8921294212341, -122.730749845505 47.8933900594711, -122.730208039284 47.894441485405, -122.729588449001 
883991900001	883991900002	883991900003	1	11.11111111	4	LINESTRING(-122.71107852459 47.8776508569717, -122.712398171425 47.8788793087006, -122.71447956562 47.8810706734657, -122.715329825878 47.8818699717522)

===== ground_truth_route.txt
Edge ID	Traversed From to To

884147800801	1

884147800802	1



In [7]:
import re
import numpy as np
import pandas as pd
from pyproj import Transformer
from shapely.geometry import LineString, box

# Seattle is in UTM zone 10N (EPSG:32610): this converts degrees to metres
TO_M = Transformer.from_crs(4326, 32610, always_xy=True)
TO_LL = Transformer.from_crs(32610, 4326, always_xy=True)

def load_gps(path):
    df = pd.read_csv(path, sep='\t', header=0)

    df.columns = df.columns.str.strip()

    df = df.rename(columns={
        'Date (UTC)': 'date',
        'Time (UTC)': 'time',
        'Latitude': 'lat',
        'Longitude': 'lon'
    })

    df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
    df['lon'] = pd.to_numeric(df['lon'], errors='coerce')

    df = df.dropna(subset=['lat', 'lon']).reset_index(drop=True)

    if df.empty:
        raise ValueError("No valid GPS coordinates found in gps_data.txt")

    df['timestamp'] = pd.to_datetime(
        df['date'].astype(str) + ' ' + df['time'].astype(str),
        format='%d-%b-%Y %H:%M:%S',
        errors='coerce'
    )

    df = df.dropna(subset=['timestamp']).reset_index(drop=True)

    if df.empty:
        raise ValueError("No valid GPS timestamps found.")

    df['t'] = (
        df['timestamp'] - df['timestamp'].iloc[0]
    ).dt.total_seconds()

    df['x'], df['y'] = TO_M.transform(
        df['lon'].to_numpy(),
        df['lat'].to_numpy()
    )

    return df


def load_roads(path):
    rows = []
    skipped = 0

    with open(path, 'r', errors='replace') as f:
        next(f, None)  # Skip header

        for line in f:
            try:
                m = re.search(r'LINESTRING\s*\((.*?)\)', line)

                if not m:
                    skipped += 1
                    continue

                head = line[:m.start()].strip().split('\t')

                if len(head) < 6:
                    skipped += 1
                    continue

                pts = []
                for p in m.group(1).split(','):
                    values = p.strip().split()

                    if len(values) >= 2:
                        pts.append(tuple(map(float, values)))

                if len(pts) < 2:
                    skipped += 1
                    continue

                # Detect longitude-first or latitude-first coordinates
                if abs(pts[0][0]) > 90:
                    lons, lats = zip(*[(p[0], p[1]) for p in pts])
                else:
                    lats, lons = zip(*[(p[0], p[1]) for p in pts])

                xs, ys = TO_M.transform(lons, lats)

                rows.append({
                    'edge_id': int(head[0]),
                    'from_node': head[1],
                    'to_node': head[2],
                    'two_way': int(head[3]),
                    'speed': float(head[4]),
                    'vertex_count': int(head[5]),
                    'geometry': LineString(zip(xs, ys))
                })

            except (ValueError, TypeError, IndexError):
                skipped += 1

    roads = pd.DataFrame(rows)

    if roads.empty:
        raise ValueError(
            "No valid roads loaded. Check the road_network.txt format."
        )

    print("Roads loaded:", len(roads))
    print("Skipped lines:", skipped)

    return roads


def load_truth(path):
    import pandas as pd

    df = pd.read_csv(path, sep='\t', header=0)

    print("Ground-truth columns:", df.columns.tolist())
    print("Raw rows:", len(df))

    # The first column contains the Edge ID
    df = df.iloc[:, :2].copy()
    df.columns = ['edge_id', 'direction']

    df['edge_id'] = pd.to_numeric(
        df['edge_id'], errors='coerce'
    )

    df = df.dropna(subset=['edge_id']).reset_index(drop=True)
    df['edge_id'] = df['edge_id'].astype('int64')

    print("Valid ground-truth rows:", len(df))

    return df


def crop_roads(roads, gps, margin=500):
    """Keep only roads inside the GPS trace's bounding box (+ margin metres)."""
    area = box(gps.x.min() - margin, gps.y.min() - margin,
               gps.x.max() + margin, gps.y.max() + margin)
    keep = roads.geometry.apply(lambda g: g.intersects(area))
    return roads[keep].reset_index(drop=True)

In [8]:
gps = load_gps(f'{DATA}/gps_data.txt')
roads_all = load_roads(f'{DATA}/road_network.txt')
truth = load_truth(f'{DATA}/ground_truth_route.txt')
roads = crop_roads(roads_all, gps)

print(len(gps), len(roads_all), len(roads), len(truth))
print(gps.head())
print(gps[['x', 'y']].describe())

Roads loaded: 158167
Skipped lines: 0
Ground-truth columns: ['Edge ID', 'Traversed From to To']
Raw rows: 576
Valid ground-truth rows: 576
7531 158167 23456 576
          date      time        lat         lon  Unnamed: 4  Unnamed: 5  \
0  17-Jan-2009  20:27:37  47.667483 -122.107083         NaN         NaN   
1  17-Jan-2009  20:27:38  47.667500 -122.107067         NaN         NaN   
2  17-Jan-2009  20:27:39  47.667500 -122.107067         NaN         NaN   
3  17-Jan-2009  20:27:40  47.667517 -122.107033         NaN         NaN   
4  17-Jan-2009  20:27:41  47.667533 -122.106983         NaN         NaN   

            timestamp    t              x             y  
0 2009-01-17 20:27:37  0.0  567034.213288  5.279730e+06  
1 2009-01-17 20:27:38  1.0  567035.438149  5.279731e+06  
2 2009-01-17 20:27:39  2.0  567035.438149  5.279731e+06  
3 2009-01-17 20:27:40  3.0  567037.924231  5.279733e+06  
4 2009-01-17 20:27:41  4.0  567041.656531  5.279735e+06  
                   x             y
count

In [9]:
missing = set(truth.edge_id) - set(roads.edge_id)
print('truth roads missing from cropped network:', len(missing))

truth roads missing from cropped network: 0


In [10]:
gps.to_pickle(f'{DATA}/gps.pkl')
roads.to_pickle(f'{DATA}/roads.pkl')

In [11]:
import numpy as np
import pandas as pd
from shapely import STRtree
from shapely.geometry import Point


def build_candidates(gps, roads, radius=50.0):
    """One row per (GPS point, nearby road)."""
    geoms = roads['geometry'].values
    edge_ids = roads['edge_id'].values
    tree = STRtree(geoms)                       # fast spatial index
    rows = []
    for i, (x, y) in enumerate(zip(gps['x'].values, gps['y'].values)):
        p = Point(x, y)
        for j in tree.query(p.buffer(radius)):
            g = geoms[j]
            d = g.distance(p)
            if d <= radius:
                rows.append((i, edge_ids[j], d, g.project(p), g.length))
    return pd.DataFrame(rows, columns=['pt_idx', 'edge_id', 'dist', 'along', 'road_len'])

In [12]:
cands = build_candidates(gps, roads, radius=50)
counts = cands.groupby('pt_idx').size().reindex(range(len(gps)), fill_value=0)
print('rows:', len(cands))
print('avg candidates per point:', counts.mean())
print('points with 0 candidates:', (counts == 0).sum())
print(counts.describe())
cands.to_pickle(f'{DATA}/candidates.pkl')

rows: 33579
avg candidates per point: 4.458770415615456
points with 0 candidates: 0
count    7531.000000
mean        4.458770
std         2.305486
min         1.000000
25%         3.000000
50%         4.000000
75%         6.000000
max        21.000000
dtype: float64


In [13]:
import numpy as np
import pandas as pd
from shapely.geometry import Point


def assign_true_roads(gps, roads, truth, window=15):
    """Give each GPS point its true road.
    Walks forward along the ground-truth route; for each point it picks the
    closest road among the next `window` roads on the route, and never goes back."""
    geom = dict(zip(roads['edge_id'], roads['geometry']))
    route = [e for e in truth['edge_id'].tolist() if e in geom]
    print('route length used:', len(route), 'of', len(truth))

    k = 0
    out_edge, out_dist = [], []
    for x, y in zip(gps['x'].values, gps['y'].values):
        p = Point(x, y)
        hi = min(k + window, len(route))
        ds = [geom[route[j]].distance(p) for j in range(k, hi)]
        best = k + int(np.argmin(ds))
        k = best
        out_edge.append(route[best])
        out_dist.append(min(ds))
    return pd.DataFrame({'pt_idx': np.arange(len(gps)),
                         'true_edge': out_edge,
                         'dist_to_true': out_dist})

In [14]:
print("GPS points:", len(gps))
print("Roads:", len(roads))
print("Ground-truth entries:", len(truth))

if truth.empty:
    print("ERROR: Ground-truth data is empty. Check load_truth().")

elif roads.empty:
    print("ERROR: Road network is empty.")

else:
    print("Ground-truth road IDs:", truth['edge_id'].head().tolist())
    print("Road network IDs:", roads['edge_id'].head().tolist())

    tp = assign_true_roads(gps, roads, truth)

    print(tp['dist_to_true'].describe())
    print(
        'Points more than 30 m from their assigned road:',
        (tp['dist_to_true'] > 30).sum()
    )

    merged = tp.merge(
        cands[['pt_idx', 'edge_id']],
        left_on=['pt_idx', 'true_edge'],
        right_on=['pt_idx', 'edge_id'],
        how='left'
    )

    found = (
        merged.groupby('pt_idx')['edge_id']
        .apply(lambda s: s.notna().any())
    )

    print(
        'Fraction of points whose true road is among candidates:',
        found.mean()
    )

    tp.to_pickle(f'{DATA}/truth_per_point.pkl')
    print("Ground-truth assignments saved.")

GPS points: 7531
Roads: 23456
Ground-truth entries: 576
Ground-truth road IDs: [884147800801, 884147800802, 884147800421, 884147800422, 884147800423]
Road network IDs: [884130300128, 884130300129, 884130300135, 884130300136, 884130300137]
route length used: 576 of 576
count    7531.000000
mean        3.892078
std         6.905103
min         0.000011
25%         1.342424
50%         2.762156
75%         4.882274
max       151.979297
Name: dist_to_true, dtype: float64
Points more than 30 m from their assigned road: 28
Fraction of points whose true road is among candidates: 0.9970787412030275
Ground-truth assignments saved.


In [15]:
import folium
import numpy as np

def ll(geom):
    xs, ys = geom.xy
    lon, lat = TO_LL.transform(np.array(xs), np.array(ys))
    return list(zip(lat, lon))

geom_of = dict(zip(roads.edge_id, roads.geometry))

m = folium.Map(
    location=[gps.lat.mean(), gps.lon.mean()],
    zoom_start=11,
    tiles="https://tile.openstreetmap.de/{z}/{x}/{y}.png",
    attr="© OpenStreetMap contributors"
)

for i in gps.sample(min(15, len(gps)), random_state=0).index:
    edge_id = tp.loc[i, 'true_edge']

    folium.PolyLine(
        ll(geom_of[edge_id]),
        color='blue',
        weight=4
    ).add_to(m)

    folium.CircleMarker(
        location=[gps.loc[i, 'lat'], gps.loc[i, 'lon']],
        radius=4,
        color='red',
        fill=True,
        popup=f'point {i}'
    ).add_to(m)

m

In [16]:
route = [e for e in truth.edge_id if e in set(roads.edge_id)]
print('last assigned road is last route road:', tp.true_edge.iloc[-1] == route[-1])
print('distinct roads used:', tp.true_edge.nunique(), 'of', len(route))

last assigned road is last route road: True
distinct roads used: 558 of 576


In [17]:
gps = gps.loc[:, ~gps.columns.str.startswith('Unnamed')]
gps.to_pickle(f'{DATA}/gps.pkl')